# Figure C1: frequency spectra of the near-bottom vertical velocity, moorings vs CROCO

One panel per mooring: power spectral density (Welch) of $w_u=-\mathbf{u}\cdot\nabla D$ at the deepest current meter (red) and of the same quantity in CROCO sampled at the same height above bottom (blue; optionally the true model $w$, green).
Based on `plot_figure4_corrected.ipynb` (same data files, same QC, same `OBS_SLOPE_KM` / `MODEL_SLOPE_KM` choices) and on the spectra of `plot_figure5.py`, with these differences:

| `plot_figure5.py` | here |
|---|---|
| a mooring is skipped (empty panel) as soon as it contains one NaN | Welch windows (30 days) with at most `MAX_NAN_FRAC` NaN (linearly interpolated) are averaged; the others are dropped. The number of windows is printed |
| IRE quality flag applied to the IRW arrays | each mooring is masked with its own flags |
| "cycle per day" = `2 f / f_M2`, i.e. a day of 89 400 s, so the tidal lines are at 2.00 (M$_2$), 2.07 (S$_2$), 0.96 (O$_1$), 1.04 (K$_1$) | true cycles per day (86 400 s): M$_2$ 1.93, S$_2$ 2.00, O$_1$ 0.93, K$_1$ 1.00, and $f=2\Omega\sin\phi$ at the mooring latitude |
| fixed time step / window assumed for the model | `MODEL_DT_H` (output interval of the model snapshots in hours) is a parameter; the window is shortened if the model record is shorter than 30 days |
| slope `rrex_dD_10km.nc`, model sampled at a hand-typed hab | SRTM15 slope (`OBS_SLOPE_KM`), model at `hab_obs` (file `rrex_w_moorings_hab.nc`) |

**Model time step**: read from the extraction file (`dt_s`, written from `simul.dt`); `MODEL_DT_H` overrides it. With the 12-hourly run the model spectrum stops at the Nyquist frequency (1 cpd), so the semi-diurnal band (M$_2$, S$_2$ near 2 cpd) and $f$ are not resolved; the hourly run (`rrexnumsb200_RSUP5_NOFILT_tracer_article_hourly`, file `rrex_w_moorings_hab_1h.nc`) reaches 12 cpd like the moorings. Select the file with `FILE_HAB`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, NullFormatter
from netCDF4 import Dataset
from pathlib import Path

# ---------------- configuration ----------------
DATA_DIR  = Path('./data')                      # folder with the three kinds of files
FILE_D    = DATA_DIR / 'rrex_dD_10km.nc'        # only used as a fallback if the new extraction file has no SRTM15 slopes
OBS_SLOPE_KM   = 10.                            # smoothing length [km] of the OBSERVED slope (SRTM15), among slope_length_km of FILE_HAB
MODEL_SLOPE_KM = 10.                            # smoothing length [km] of the MODEL slope (0 = grid scale), chosen independently of the obs
FILE_CROCO= DATA_DIR / 'rrex_bottom_w_mooring_notracer.nc'   # old extraction (hand-typed hab list)
FILE_HAB  = DATA_DIR / 'rrex_w_moorings_hab_1h.nc'           # extraction used for the first figure: hourly run (use 'rrex_w_moorings_hab.nc' for the 12-hourly run)
MOORINGS  = ['IRW', 'IRM', 'IRE', 'RRT', 'ICW', 'ICM', 'ICE']    # index in dDdx, wb*, wub* = position in this list

# model quantities to plot (the first is the main one; the others are drawn on top in other colours).  New file:
#   'wu_L' : -u.grad(H smoothed over MODEL_SLOPE_KM)     'wu' : -u.grad(H), grid-scale slope     'w' : true model vertical velocity
# Old file (no new extraction): 'wu_L'/'wu' -> 'wub', 'w' -> 'wb'.
MODEL_SERIES = ['wu_L', 'w']                                  # keep both: figure 1 uses 'wu_L' (or 'w' if MOORING_VAR='WCUR'), figure 2 uses 'wu_L' and 'w'
MOORING_VAR = 'wu'      # 'wu' : -u.grad(D) from the deepest current meter ; 'WCUR': measured vertical velocity
QC_MAX      = 2         # keep samples with QC flag <= QC_MAX (1 = good, 2 = probably good)

# surveyed water depth at the mooring [m] (depth_cv of the old extraction script): reference for the observed hab
DEPTH_REF   = np.array([2160., 1570., 1580., 1528., 2162., 2151., 2379.])
# hab list typed by hand in the OLD extraction (what the old file was sampled at)
HAB_EXTRACT = np.array([100, 93, 116, 81, 57, 46, 282])

LOG_Y = True            # True: logarithmic pdf axis (shows the tails); False: linear axis (figures saved without the '_log' suffix)
YMIN, YMAX = 5e-5, 0.1   # y limits of the log axis, in probability density [(m/day)^-1]
YLIM_LIN = None          # y limits of the linear axis, e.g. (0, 0.03); None = automatic

dw  = 2.                                            # bin width [m/day]
w_e = np.arange(-750, 750 + dw, dw)
XLIM = (-310, 310)
SEC_PER_DAY = 86400.
fs = 14

# ---------------- spectra ----------------
SPEC_SERIES   = ['wu_L']      # model series drawn: 'wu_L' (-u.grad H smoothed over MODEL_SLOPE_KM), 'wu' (grid scale) and/or 'w' (true model w)
MODEL_DT_H    = None          # time step of the model snapshots [hours]: None = read from the extraction file (attribute dt_s; 12 h if absent)
OBS_DT_H      = 1.0           # time step of the mooring files [hours]
NPERSEG_DAYS  = 30.           # Welch window length [days] (shortened if the record is shorter)
SEASON        = (10, 11)          # None: whole mooring record.  (9, 3): only the months September to March (both included) of each year, e.g. (9, 3) = Sep-Mar; mooring only
SHOW_CI       = True          # draw the confidence interval of the spectra (shaded band)
CI_LEVEL      = 0.95          # confidence level
MAX_NAN_FRAC  = 0.02          # a window is kept if at most this fraction of its samples is NaN (they are linearly interpolated)
OMEGA         = 7.2921e-5     # Earth rotation rate [rad s-1]
YLIM_PSD      = (1e-7, 10)
FS_PLOT       = 20
MONTH_NAMES   = 'Jan Feb Mar Apr May Jun Jul Aug Sep Oct Nov Dec'.split()
SEASON_TAG    = '' if SEASON is None else '_%s%s' % (MONTH_NAMES[SEASON[0] - 1], MONTH_NAMES[SEASON[1] - 1])    # suffix of the saved figures
MOOR_LABEL    = 'Mooring' + ('' if SEASON is None else ' (%s-%s)' % (MONTH_NAMES[SEASON[0] - 1], MONTH_NAMES[SEASON[1] - 1]))
MODEL_SERIES = list(SPEC_SERIES)    # series loaded from the extraction file

# ---------------- second figure: 12-hourly and hourly model output on the same panels ----------------
FILE_HAB_12H  = DATA_DIR / 'rrex_w_moorings_hab.nc'          # extraction from rrexnumsb200_RSUP5_NOFILT_tracer (12 h output)
FILE_HAB_1H   = DATA_DIR / 'rrex_w_moorings_hab_1h.nc'       # extraction from ..._tracer_article_hourly (1 h output)
COMPARE_SERIES = 'wu_L'                                      # model quantity compared: 'wu_L' (MODEL_SLOPE_KM), 'wu' (grid scale) or 'w'


In [ ]:
def masked_to_nan(a):
    """netCDF masked array -> float array with NaN for masked / fill values."""
    a = np.ma.filled(np.ma.asarray(a).astype(float), np.nan)
    a[np.abs(a) > 1e5] = np.nan                  # 999999 fill values
    return a

def read_mooring(name, i_grad, grad):
    ds = Dataset(DATA_DIR / ('RREX_%s_CURR_hourly.nc' % name))
    depth = masked_to_nan(ds['DEPTH'][:])
    zmed  = np.nanmedian(depth, axis=0)
    k     = int(np.nanargmax(zmed))                   # deepest bin
    out   = dict(name=name, k=k, nlev=depth.shape[1], k_is_last=(k == depth.shape[1] - 1), z=zmed[k])
    def get(var):
        x = masked_to_nan(ds[var][:, k])
        if var + '_QC' in ds.variables:
            q = np.ma.filled(ds[var + '_QC'][:, k], 9)
            x[q > QC_MAX] = np.nan
        return x
    u, v = get('UCUR'), get('VCUR')
    out['u'], out['v'] = u, v                                          # [m s-1] for the horizontal-velocity spectra
    out['wu']   = -(u * grad['dDdx'][i_grad] + v * grad['dDdy'][i_grad]) * SEC_PER_DAY   # D > 0: bottom at z = -D
    out['WCUR'] = get('WCUR') * SEC_PER_DAY
    out['lat'] = float(np.ravel(masked_to_nan(ds['LATITUDE'][:]))[0])
    # optional season: samples outside the months SEASON = (first, last) (both included, may wrap over the new year) -> NaN
    days = np.ma.filled(ds['TIME'][:], np.nan).astype(float)                        # days since 1950-01-01
    month = (np.datetime64('1950-01-01') + (days * 86400).astype('timedelta64[s]')).astype('datetime64[M]').astype(int) % 12 + 1
    out['time'], out['month'] = days, month
    if SEASON is not None:
        m1, m2 = SEASON
        keep = (month >= m1) & (month <= m2) if m1 <= m2 else (month >= m1) | (month <= m2)
        for key in ('u', 'v', 'wu', 'WCUR'):
            out[key] = np.where(keep, out[key], np.nan)
        out['season_frac'] = float(keep.mean())
    ds.close()
    return out

# observed bottom slope at the moorings: SRTM15 slope smoothed over OBS_SLOPE_KM, computed by the extraction script
if FILE_HAB.exists() and 'dDdx_topo15' in Dataset(FILE_HAB).variables:
    with Dataset(FILE_HAB) as nc:
        L = np.asarray(nc['slope_length_km'][:], float)
        k = int(np.argmin(abs(L - OBS_SLOPE_KM)))
        grad = {'dDdx': np.asarray(nc['dDdx_topo15'][k], float), 'dDdy': np.asarray(nc['dDdy_topo15'][k], float),
                'D': np.asarray(nc['D_topo15'][:], float)}
    print('observed slope: SRTM15 smoothed over %g km (from %s)' % (L[k], FILE_HAB.name))
else:
    with Dataset(FILE_D) as nd:                                 # fallback: precomputed slopes (method not documented)
        grad = {k: np.asarray(nd[k][:], float) for k in ('D', 'dDdx', 'dDdy')}
    print('observed slope: %s (no SRTM15 slopes found)' % FILE_D.name)
obs = [read_mooring(m, i, grad) for i, m in enumerate(MOORINGS)]

USE_NEW = FILE_HAB.exists()
mod, mod_label = {}, {}                                          # mod[name] = list over moorings of the model series
if USE_NEW:
    with Dataset(FILE_HAB) as nc:
        Lm = np.asarray(nc['slope_length_km'][:], float) if 'slope_length_km' in nc.variables else np.array([])
        for name in MODEL_SERIES:
            if name == 'w':
                arr, mod_label[name] = nc['w_obs_hab'][:], r'CROCO: $w$'
            elif name == 'wu' or (name == 'wu_L' and MODEL_SLOPE_KM == 0 and 'wu_L_obs_hab' not in nc.variables):
                arr, mod_label[name] = nc['wu_obs_hab'][:], r'CROCO: $-\mathbf{u}\cdot\nabla H$ (grid scale)'
            elif 'wu_L_obs_hab' in nc.variables:
                kL = int(np.argmin(abs(Lm - MODEL_SLOPE_KM)))
                assert abs(Lm[kL] - MODEL_SLOPE_KM) < 1e-6, 'MODEL_SLOPE_KM=%g not in %s' % (MODEL_SLOPE_KM, Lm)
                arr, mod_label[name] = nc['wu_L_obs_hab'][:, :, kL], r'CROCO: $-\mathbf{u}\cdot\nabla H$'
            else:                                                # file from the previous version of the extraction
                print('wu_L_obs_hab not in %s: using wu_s (smoothing %g km)' % (FILE_HAB.name, nc.smooth_km))
                arr, mod_label[name] = nc['wu_s_obs_hab'][:], r'CROCO: $-\mathbf{u}\cdot\nabla H$'
            arr = np.asarray(np.ma.filled(np.ma.asarray(arr).astype(float), np.nan), float)    # (time, mooring)
            mod[name] = [arr[:, i] for i in range(len(MOORINGS))]
        hab_model = np.asarray(nc['hab_obs'][:], float)           # hab at which the model was sampled
        h_model   = np.asarray(nc['h'][:], float)
    print('model file: %s (sampled at hab_obs), series %s' % (FILE_HAB.name, ', '.join(MODEL_SERIES)))
else:
    with Dataset(FILE_CROCO) as nc:
        for name in MODEL_SERIES:
            old = 'wb' if name == 'w' else 'wub'
            mod[name] = [np.asarray(nc['%s%d' % (old, i)][:], float) for i in range(len(MOORINGS))]
            mod_label[name] = r'CROCO: $w$' if name == 'w' else r'CROCO: $-\mathbf{u}\cdot\nabla H$'
    hab_model = HAB_EXTRACT.astype(float)
    h_model   = None
    print('%s not found: using the OLD file %s, sampled at the hand-typed hab list' % (FILE_HAB.name, FILE_CROCO.name))


In [ ]:
def clean(x): return x[np.isfinite(x)]
print('%-4s %8s %10s | %8s | %10s %9s' % ('', 'z_sensor', 'depth_ref', 'hab_obs', 'hab model', 'model-obs'))
for i, o in enumerate(obs):
    o['hab'] = DEPTH_REF[i] - o['z']                  # height of the deepest bin above the surveyed bottom
    d = hab_model[i] - o['hab']
    print('%-4s %8.1f %10.0f | %8.1f | %10.1f %9.1f' % (o['name'], o['z'], DEPTH_REF[i], o['hab'], hab_model[i], d))
print()
print('hab_obs   = depth_ref - z_sensor : height of the deepest current meter above the surveyed bottom')
print('hab model = height above the MODEL floor at which the model series was sampled')
bad = [o['name'] for i, o in enumerate(obs) if abs(hab_model[i] - o['hab']) > 5]
if bad:
    print('\n!! model and mooring are NOT at the same hab (> 5 m apart) for: %s' % ', '.join(bad))
    print('   -> run extract_bottom_w_moorings_croco_hab.py, which samples the model at hab_obs')
else:
    print('\nmodel and mooring are at the same height above bottom (within 5 m) at all moorings')

In [ ]:
import scipy.signal as sig
from scipy.stats import chi2


class Spec(tuple):
    """(frequency, psd) that also carries `nu`, the equivalent degrees of freedom of the estimate (for the confidence interval)."""
    def __new__(cls, fp, nu):
        o = super().__new__(cls, fp); o.nu = nu; return o


def edof(K, mult=1):
    """Equivalent degrees of freedom of a Welch estimate averaging K segments with 50 % overlap and a Hann window (Welch 1967):
    nu = 36 K^2 / (19 K - 1)  (~1.9 K for large K).  `mult` = 2 for the sum of two independent spectra (PSD_u + PSD_v)."""
    return mult * 36. * K ** 2 / (19. * K - 1.) if K > 0 else np.nan


def ci_band(ax, sp, color):
    """Shaded CI_LEVEL confidence interval of the spectrum `sp` (chi-square law with sp.nu degrees of freedom, multiplicative on a log axis)."""
    nu = getattr(sp, 'nu', np.nan)
    if not SHOW_CI or not np.isfinite(nu):
        return
    a = 1. - CI_LEVEL
    lo, hi = nu / chi2.ppf(1. - a / 2., nu), nu / chi2.ppf(a / 2., nu)
    f, p = sp
    ax.fill_between(cpd(f[1:]), p[1:] * lo, p[1:] * hi, color=color, alpha=0.2, lw=0)


def welch_gappy(x, dt_s, nperseg_days):
    """Welch spectrum [x^2 Hz^-1] of a series with gaps: non-overlapping windows with at most MAX_NAN_FRAC NaN are
    interpolated and averaged (Hann window, mean removed).  The window is shortened to a quarter of the record if the record is
    shorter than nperseg_days.  Returns (freq [Hz], psd, number of windows, window length [days])."""
    x = np.asarray(x, float)
    n = int(round(nperseg_days * 86400. / dt_s))
    if len(x) < 2 * n:                                   # record too short: use windows of a quarter of the record
        n = max(len(x) // 4, 8)
    specs = []
    for i0 in range(0, len(x) - n + 1, n // 2):          # 50 % overlap
        w = x[i0:i0 + n].copy()
        bad = ~np.isfinite(w)
        if bad.mean() > MAX_NAN_FRAC or bad.all():
            continue
        if bad.any():
            w[bad] = np.interp(np.flatnonzero(bad), np.flatnonzero(~bad), w[~bad])
        f, p = sig.welch(w, 1. / dt_s, window='hann', nperseg=n, detrend='constant')
        if n % 2 == 0:                                   # scipy does not double the Nyquist bin of a one-sided spectrum -> correct it
            p = p.copy(); p[-1] *= 2.
        specs.append(p)
    if not specs:
        return np.array([np.nan]), np.array([np.nan]), 0, n * dt_s / 86400.
    return f, np.mean(specs, axis=0), len(specs), n * dt_s / 86400.

dt_file = None
if USE_NEW:
    with Dataset(FILE_HAB) as nc:
        dt_file = getattr(nc, 'dt_s', None)
dt_mod = MODEL_DT_H * 3600. if MODEL_DT_H else (float(dt_file) if dt_file else 12 * 3600.)
dt_obs = OBS_DT_H * 3600.
print('model time step: %g h (%s) -> Nyquist %.2f cpd' % (dt_mod / 3600., 'file' if (dt_file and not MODEL_DT_H) else 'default/config', 43200. / dt_mod))
spec_obs, spec_mod = [], {n: [] for n in SPEC_SERIES}
print('%-4s %-8s %7s %8s %9s' % ('', 'series', 'samples', 'windows', 'window[d]'))
for i, o in enumerate(obs):
    x = o[MOORING_VAR] / SEC_PER_DAY                       # m/day -> m/s
    f, p, nw, wd = welch_gappy(x, dt_obs, NPERSEG_DAYS)
    spec_obs.append(Spec((f, p), edof(nw))); print('%-4s %-8s %7d %8d %9.1f' % (o['name'], 'mooring', len(x), nw, wd))
    for name in SPEC_SERIES:
        xm = mod[name][i] / SEC_PER_DAY
        f, p, nw, wd = welch_gappy(xm, dt_mod, NPERSEG_DAYS)
        spec_mod[name].append(Spec((f, p), edof(nw))); print('%-4s %-8s %7d %8d %9.1f' % ('', name, len(xm), nw, wd))


In [ ]:
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
fsz = FS_PLOT
DAY = 86400.
# tidal frequencies [cycles per day, true 86400 s day]
TIDES = {'O$_1$': 1. / 92950., 'K$_1$': 1. / 86164., 'M$_2$': 1. / 44700., 'S$_2$': 1. / 43200.}
cpd = lambda f_hz: np.asarray(f_hz) * DAY
line_o, line_m = ('tab:red', 'tab:blue'), None
STYLE = {'wu_L': 'tab:blue', 'wu': 'tab:purple', 'w': 'tab:green'}

PERIODS = {'1 month': 1. / 30.44, '1 week': 1. / 7.}                  # [cycles per day]


def decorate(ax, lat, title, ylim=None):
    """Tidal / inertial lines, limits, grid and labels of one spectrum panel (lat: latitude for f)."""
    yl = ylim or YLIM_PSD
    fi = 2 * OMEGA * np.sin(np.deg2rad(lat)) / (2 * np.pi)                # inertial frequency [Hz]
    for lab, fr in list(TIDES.items()) + [('$f$', fi)]:
        ax.axvline(cpd(fr), color='k', ls='--', lw=0.8)
    ax.text(cpd(fi), yl[1] * 0.6, '$f$', ha='right', va='top', fontsize=0.8 * fsz)
    ax.text(0.5 * (cpd(TIDES['O$_1$']) + cpd(TIDES['K$_1$'])), yl[1] * 0.6, 'O$_1$,K$_1$', ha='center', va='top', fontsize=0.8 * fsz)
    ax.text(0.5 * (cpd(TIDES['M$_2$']) + cpd(TIDES['S$_2$'])) * 1.02, yl[1] * 0.6, 'M$_2$,S$_2$', ha='left', va='top', fontsize=0.8 * fsz)
    for lab, per in PERIODS.items():                                      # monthly / weekly periods [cpd]
        ax.axvline(per, color='tab:gray', ls=':', lw=1.3)
        ax.text(per, yl[0] * 1.5, lab, rotation=90, ha='right', va='bottom', fontsize=0.65 * fsz, color='dimgray')
    ax.set_ylim(*yl); ax.set_xlim(0.025, 14)
    ax.grid(True, which='major', alpha=0.5)
    ax.grid(True, which='minor', alpha=0.2, lw=0.5)                       # finer grid (2..9 x 10^n)
    ax.set_xlabel('Cycle per day', fontsize=fsz); ax.set_ylabel(r'PSD [(m s$^{-1}$)$^2$ Hz$^{-1}$]', fontsize=fsz)
    ax.set_title(title, fontsize=fsz, loc='left')
    ax.tick_params(labelsize=fsz)


def mean_spec(specs):
    """Arithmetic mean of the PSD over the moorings (all spectra share the same frequencies; empty ones are ignored).
    The degrees of freedom of the mean are the sum of those of the spectra (sites taken as independent)."""
    f = max((s[0] for s in specs), key=len)
    used = [s for s in specs if len(s[1]) == len(f) and np.isfinite(s[1]).any()]
    return Spec((f, np.nanmean([s[1] for s in used], axis=0)), sum(s.nu for s in used))


LAT_MEAN = np.mean([o['lat'] for o in obs])
FS_LAB, FS_TICK = 0.7 * fsz, 0.8 * fsz                       # smaller axis labels (units) and tick labels, used for all three figures


def small_labels(ax):
    ax.xaxis.label.set_size(FS_LAB); ax.yaxis.label.set_size(FS_LAB)
    ax.tick_params(labelsize=FS_TICK)


fig = plt.figure(figsize=(20, 14))
gs = gridspec.GridSpec(3, 6, hspace=0.35, wspace=0.9)      # 6 columns: each panel spans 2, the average sits in the middle of the empty space
for i, o in enumerate(obs):
    ax = fig.add_subplot(gs[i // 3, 2 * (i % 3):2 * (i % 3) + 2])
    sp = spec_obs[i]; f, p = sp
    ax.loglog(cpd(f[1:]), p[1:], color='tab:red', lw=1.2, label=MOOR_LABEL)
    ci_band(ax, sp, 'tab:red')
    for name in SPEC_SERIES:
        sp = spec_mod[name][i]; f, p = sp
        ax.loglog(cpd(f[1:]), p[1:], color=STYLE[name], lw=1.2, alpha=0.75, label=mod_label[name])
        ci_band(ax, sp, STYLE[name])
    decorate(ax, o['lat'], '%s) %s' % ('abcdefg'[i], o['name'])); small_labels(ax)
    if i == 0:
        leg_h, leg_l = ax.get_legend_handles_labels()      # legend is drawn in the empty space (below)
# panel h): average over all sites
ax = fig.add_subplot(gs[2, 4:6])
sp = mean_spec(spec_obs); f, p = sp
ax.loglog(cpd(f[1:]), p[1:], color='tab:red', lw=1.5, label=MOOR_LABEL)
ci_band(ax, sp, 'tab:red')
for name in SPEC_SERIES:
    sp = mean_spec(spec_mod[name]); f, p = sp
    ax.loglog(cpd(f[1:]), p[1:], color=STYLE[name], lw=1.5, alpha=0.75, label=mod_label[name])
    ci_band(ax, sp, STYLE[name])
decorate(ax, LAT_MEAN, 'h) Average over all sites'); small_labels(ax)
axl = fig.add_subplot(gs[2, 2:4]); axl.axis('off')                          # empty space between g) and h)
axl.legend(leg_h, leg_l, loc='center', fontsize=0.9 * fsz, frameon=True)
fig.savefig('figure_appendixC_corrected' + SEASON_TAG + '.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# --- Same spectra with the 12-hourly and the hourly model runs together (mooring, CROCO 12 h, CROCO 1 h) ---
def read_model_series(path, name, dt_default):
    """Model series (m/day) at each mooring from an extraction file, its time step [s] and a label."""
    with Dataset(path) as nc:
        dt = float(getattr(nc, 'dt_s', dt_default))
        if name == 'w':
            arr, lab = nc['w_obs_hab'][:], r'$w$'
        elif name == 'wu':
            arr, lab = nc['wu_obs_hab'][:], r'$-\mathbf{u}\cdot\nabla H$'
        elif 'wu_L_obs_hab' in nc.variables:
            Lm = np.asarray(nc['slope_length_km'][:], float)
            kL = int(np.argmin(abs(Lm - MODEL_SLOPE_KM)))
            arr, lab = nc['wu_L_obs_hab'][:, :, kL], r'$-\mathbf{u}\cdot\nabla H$'
        else:
            arr, lab = nc['wu_s_obs_hab'][:], r'$-\mathbf{u}\cdot\nabla H$'
    arr = np.asarray(np.ma.filled(np.ma.asarray(arr).astype(float), np.nan), float)
    return [arr[:, i] for i in range(len(MOORINGS))], dt, lab

runs = []                                                    # (label, colour, spectra per mooring)
for path, dt0, col, tag in ((FILE_HAB_12H, 12 * 3600., 'tab:blue', '12-hourly output'), (FILE_HAB_1H, 3600., 'tab:orange', 'hourly output')):
    if not Path(path).exists():
        print('missing', path, '-> not plotted'); continue
    series, dtm, lab = read_model_series(path, COMPARE_SERIES, dt0)
    spectra = []
    for i, o in enumerate(obs):
        f, p, nw, wd = welch_gappy(series[i] / SEC_PER_DAY, dtm, NPERSEG_DAYS)
        spectra.append(Spec((f, p), edof(nw)))
    print('%s: dt = %g h, %d snapshots, %d windows of %.0f days (mooring %s)' % (Path(path).name, dtm / 3600., len(series[0]), nw, wd, obs[-1]['name']))
    runs.append(('CROCO ' + lab + ', ' + tag, col, spectra))

fig = plt.figure(figsize=(20, 14))
gs = gridspec.GridSpec(3, 6, hspace=0.35, wspace=0.9)
for i, o in enumerate(obs):
    ax = fig.add_subplot(gs[i // 3, 2 * (i % 3):2 * (i % 3) + 2])
    sp = spec_obs[i]; f, p = sp
    ax.loglog(cpd(f[1:]), p[1:], color='tab:red', lw=1.2, label=MOOR_LABEL)
    ci_band(ax, sp, 'tab:red')
    for lab, col, spectra in runs:
        sp = spectra[i]; f, p = sp
        ax.loglog(cpd(f[1:]), p[1:], color=col, lw=1.2, alpha=0.75, label=lab)
        ci_band(ax, sp, col)
    decorate(ax, o['lat'], '%s) %s' % ('abcdefg'[i], o['name'])); small_labels(ax)
    if i == 0:
        leg_h, leg_l = ax.get_legend_handles_labels()      # legend is drawn in the empty space (below)
# panel h): average over all sites
ax = fig.add_subplot(gs[2, 4:6])
sp = mean_spec(spec_obs); f, p = sp
ax.loglog(cpd(f[1:]), p[1:], color='tab:red', lw=1.5, label=MOOR_LABEL)
ci_band(ax, sp, 'tab:red')
for lab, col, spectra in runs:
    sp = mean_spec(spectra); f, p = sp
    ax.loglog(cpd(f[1:]), p[1:], color=col, lw=1.5, alpha=0.75, label=lab)
    ci_band(ax, sp, col)
decorate(ax, LAT_MEAN, 'h) Average over all sites'); small_labels(ax)
axl = fig.add_subplot(gs[2, 2:4]); axl.axis('off')                          # empty space between g) and h)
axl.legend(leg_h, leg_l, loc='center', fontsize=0.9 * fsz, frameon=True)
fig.savefig('figure_appendixC_12h_vs_1h' + SEASON_TAG + '.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# --- Horizontal velocity at the deepest current meter: PSD(u) + PSD(v), mooring vs CROCO (12 h and 1 h runs) ---
def read_model_uv(path, dt_default):
    """u, v [m s-1] of an extraction file interpolated at hab_obs of each mooring: arrays (snapshot, mooring), and dt [s]."""
    with Dataset(path) as nc:
        dt = float(getattr(nc, 'dt_s', dt_default))
        hab = np.asarray(nc['hab'][:], float); hab_obs_m = np.asarray(nc['hab_obs'][:], float)
        out = []
        for var in ('u', 'v'):
            F = np.asarray(np.ma.filled(np.ma.asarray(nc[var][:]).astype(float), np.nan), float)      # (snapshot, mooring, hab)
            out.append(np.array([[np.interp(hab_obs_m[m], hab, F[n, m], left=np.nan, right=np.nan)
                                  for m in range(F.shape[1])] for n in range(F.shape[0])]))
    return out[0], out[1], dt

def psd_uv(u, v, dt):
    """PSD(u) + PSD(v) [(m s-1)^2 Hz-1] with welch_gappy; windows where either component is unusable are dropped."""
    fu, pu, nwu, wd = welch_gappy(u, dt, NPERSEG_DAYS)
    fv, pv, nwv, _ = welch_gappy(v, dt, NPERSEG_DAYS)
    return fu, pu + pv, min(nwu, nwv), wd

uv_obs = []
for o in obs:
    f, p, nw, wd = psd_uv(o['u'], o['v'], dt_obs); uv_obs.append(Spec((f, p), edof(nw, 2)))
uv_runs = []
for path, dt0, col, tag in ((FILE_HAB_1H, 3600., 'tab:orange', 'hourly output'),):          # hourly run only (the 12-hourly run cannot resolve the tidal/inertial band)
    if not Path(path).exists():
        print('missing', path, '-> not plotted'); continue
    um, vm, dtm = read_model_uv(path, dt0)
    spectra = []
    for i in range(len(obs)):
        f, p, nw, wd = psd_uv(um[:, i], vm[:, i], dtm); spectra.append(Spec((f, p), edof(nw, 2)))
    print('%s: dt = %g h, %d windows of %.0f days (last mooring)' % (Path(path).name, dtm / 3600., nw, wd))
    uv_runs.append(('CROCO, ' + tag, col, spectra))

pmax = np.nanmax([np.nanmax(p[1:]) for f, p in uv_obs])
YLIM_UV = (pmax * 1e-6, pmax * 50)                          # y limits of the u,v PSD [(m s-1)^2 Hz-1], relative to the largest mooring peak
YLAB_UV = r'PSD$_u$ + PSD$_v$ [(m s$^{-1}$)$^2$ Hz$^{-1}$]'
fig = plt.figure(figsize=(20, 14))
gs = gridspec.GridSpec(3, 6, hspace=0.35, wspace=0.9)
for i, o in enumerate(obs):
    ax = fig.add_subplot(gs[i // 3, 2 * (i % 3):2 * (i % 3) + 2])
    sp = uv_obs[i]; f, p = sp
    ax.loglog(cpd(f[1:]), p[1:], color='tab:red', lw=1.2, label=MOOR_LABEL)
    ci_band(ax, sp, 'tab:red')
    for lab, col, spectra in uv_runs:
        sp = spectra[i]; f, p = sp
        ax.loglog(cpd(f[1:]), p[1:], color=col, lw=1.2, alpha=0.75, label=lab)
        ci_band(ax, sp, col)
    decorate(ax, o['lat'], '%s) %s' % ('abcdefg'[i], o['name']), ylim=YLIM_UV)
    ax.set_ylabel(YLAB_UV)
    small_labels(ax)
    if i == 0:
        leg_h, leg_l = ax.get_legend_handles_labels()      # legend is drawn in the empty space (below)
ax = fig.add_subplot(gs[2, 4:6])
sp = mean_spec(uv_obs); f, p = sp
ax.loglog(cpd(f[1:]), p[1:], color='tab:red', lw=1.5, label=MOOR_LABEL)
ci_band(ax, sp, 'tab:red')
for lab, col, spectra in uv_runs:
    sp = mean_spec(spectra); f, p = sp
    ax.loglog(cpd(f[1:]), p[1:], color=col, lw=1.5, alpha=0.75, label=lab)
    ci_band(ax, sp, col)
decorate(ax, LAT_MEAN, 'h) Average over all sites', ylim=YLIM_UV)
ax.set_ylabel(YLAB_UV)
small_labels(ax)
axl = fig.add_subplot(gs[2, 2:4]); axl.axis('off')                          # empty space between g) and h)
axl.legend(leg_h, leg_l, loc='center', fontsize=0.9 * fsz, frameon=True)
fig.savefig('figure_appendixC_uv' + SEASON_TAG + '.png', dpi=200, bbox_inches='tight')
plt.show()
